# REWIRE quick start (Google Colab / Kaggle)

Write-only continual learning for recurring classes in spiking neural networks.

This notebook runs the checks and a small real experiment of the paper on a free GPU runtime. **Runtime → Change runtime type → GPU** (a T4 is enough; the checks also run on CPU).

What it does:
1. clone the repository and install the few dependencies (PyTorch is pre-installed on Colab and Kaggle);
2. run the property checks (Propositions 1 and 2, the solver, the gate) and the end-to-end self-test on synthetic data (no download);
3. download MNIST automatically and run REWIRE, REWIRE-gated, HLOP and rehearsal on one class-incremental stream with repetition (seed 0);
4. print the leaderboard.

The full campaign of the paper (6 datasets × 29 learners × 5 seeds) took about seven hours on one RTX A6000; see `REPRODUCE.md` for it.

In [ ]:
!git clone --depth 1 https://github.com/Aftab-Hussain302/REWIRE-Write-Only-Continual-Learning-for-Recurring-Classes-in-Spiking-Neural-Networks.git rewire_repo
%cd rewire_repo
!pip install -q -r requirements.txt
import torch; print('torch', torch.__version__, '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none (CPU mode)')

## 1. Checks (about five minutes, no data needed)
`rewire.preflight` verifies the mathematical properties the method relies on; `selftest.py` runs every learner of the paper end to end on a short synthetic stream; `pytest` covers both propositions, the LDA readout against NumPy, the recursive solve and the gate.

In [ ]:
!python3 -m rewire.preflight
!python3 -m pytest tests -q

In [ ]:
!python3 selftest.py

## 2. A real stream: MNIST, seed 0, four learners (a few minutes on a GPU)
The data are downloaded automatically by `scripts/prepare_data.py` (torchvision). The stream follows the paper's protocol: 8 classes admitted, 2 held out, 24 recurrence chunks, 80 + 40 samples per chunk.

In [ ]:
!python3 scripts/prepare_data.py --data_root /content/data --datasets mnist
!python3 run_campaign.py --datasets mnist --seeds 0 --arms rewire rewire_gated hlop replay --no-sweeps --data_root /content/data --out results/colab_demo

In [ ]:
import pandas as pd, glob
p = glob.glob('results/colab_demo/mnist/shift_none/summary_table.csv')[0]
df = pd.read_csv(p)
df[['label','ACC','BWT','mJ','mJ_readmission','skipped','AUROC_stream']].round(2)

## 3. Going further
* **Other datasets.** `scripts/prepare_data.py --data_root /content/data --datasets nmnist shd` downloads N-MNIST (~1 GB) and SHD (~0.6 GB) from their official servers; CIFAR-10/100 are fetched by torchvision and passed through the frozen ResNet-18 (a GPU makes the feature cache a matter of minutes).
* **Tiny-ImageNet needs more RAM** than a free Colab/Kaggle runtime usually offers: the loader holds 100 000 images of 64×64 pixels in memory (1.2 GB) and extracts ResNet-18 features from them. On a free runtime use the smaller streams, or reduce the per-class sample count, e.g. `--set n_train_per_class=200 n_test_per_class=50`.
* **All five seeds and both sweeps** for one dataset: drop `--seeds 0 --arms ... --no-sweeps`. **Every learner of the paper** on every dataset: `scripts/run_all.py` (the parallel launcher; it resumes if the runtime is interrupted).
* **The paper's tables and figures from the released records** (no training): `python3 scripts/unpack_records.py results/campaign_2026-10-09_1120` then `python3 plotting/make_plots.py --run results/campaign_2026-10-09_1120 --out plotting/assets_rebuilt`.